# Magnetic Field and NV Drift

In [7]:
%load_ext autoreload
%autoreload 2

from copy import copy
import qickdawg as qd

from importlib.metadata import version
version('numpy')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


'2.3.3'

In [8]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from datetime import datetime
import time

from scipy.optimize import curve_fit

In [9]:
qd.start_client('192.168.3.1')

In [10]:
default_config = qd.NVConfiguration()

default_config.adc_channel = 0
default_config.edge_counting = True
default_config.high_threshold = 8000
default_config.low_threshold = 500


default_config.mw_channel = 0
default_config.mw_nqz = 1
default_config.mw_gain = 12000

default_config.laser_gate_pmod = 0

default_config.relax_delay_tns = 50 # between each rep, wait for everything to catch up, mostly aom

In [11]:
qd.laser_on(default_config)

0

In [12]:
config_PL = copy(default_config)

config_PL.readout_integration_treg = 2**16-1 # Maxium number of integrated points
config_PL.reps = 10000

prog_PL = qd.PLIntensity(config_PL) 

In [13]:
config_ODMR = copy(default_config)

config_ODMR.readout_integration_tus = qd.max_int_time_tus

config_ODMR.mw_gain = 12000

config_ODMR.pre_init = True 
config_ODMR.reps = 2000
config_ODMR.relax_delay_treg = 300

config_ODMR.add_linear_sweep('mw', 'fMHz', start=2100, stop=2400, delta=0.5)

prog_ODMR = qd.LockinODMR(config_ODMR)


Requested 2100 to 2400 by 0.5
Instead using 2100.0 to 2400.000228881836 by 0.5000003814697266 in 601 steps


In [15]:
config_ODMR.relax_delay_treg = 1
config_ODMR.relax_delay_tns/16

0.20345052083333334

In [35]:
def save_odmr_csv(run_dir, index, freqs, signal, reference, cps, timestamp):
    df = pd.DataFrame({
        "frequency_MHz": freqs,
        "signal": signal,
        "reference": reference,
        "ratio": signal / reference,
        "cps_prior": cps,
        "timestamp": timestamp
    })
    
    fname = os.path.join(run_dir, f"odmr_{index:03d}.csv")
    df.to_csv(fname, index=False)

In [26]:
def lorentzian(f, baseline, log_amp, f0, gamma):
    A = np.exp(log_amp)  # enforce A > 0
    return baseline - A / (1 + ((f - f0)/gamma)**2)

def fit_lorentzian(freqs, contrast):
    baseline0 = np.median(contrast)
    amp0 = np.log(np.max(contrast) - np.min(contrast) + 1e-6)
    f0_0 = freqs[np.argmin(contrast)]
    gamma0 = (freqs[-1] - freqs[0]) / 20

    p0 = [baseline0, amp0, f0_0, gamma0]

    try:
        params, cov = curve_fit(
            lorentzian,
            freqs,
            contrast,
            p0=p0,
            maxfev=5000
        )
        baseline, log_amp, f0, gamma = params

        # Evaluate fitted peak value
        peak_val = lorentzian(f0, *params)

        return {
            "success": True,
            "f0": f0,
            "peak_val": peak_val,
            "params": params,
            "cov": cov
        }

    except Exception:
        # Fallback: use data minimum
        idx = np.argmin(contrast)
        return {
            "success": False,
            "f0": freqs[idx],
            "peak_val": contrast[idx],
            "params": None,
            "cov": None
        }

def plot_single_odmr(run_dir, index, freqs, signal, reference, cps, timestamp):
    contrast = signal / reference

    # Fit
    fit = fit_lorentzian(freqs, contrast)
    peak_freq = fit["f0"]
    peak_val = fit["peak_val"]

    # Plot raw data
    plt.figure(figsize=(7,5))
    plt.plot(freqs, contrast, 'k.', label="Data")

    # Overlay fit if available
    if fit["success"]:
        f_fit = np.linspace(freqs.min(), freqs.max(), 1000)
        y_fit = lorentzian(f_fit, *fit["params"])
        plt.plot(f_fit, y_fit, 'r-', label="Lorentzian fit")

    # Mark fitted peak
    plt.annotate(
        f"{peak_freq:.2f} MHz",
        xy=(peak_freq, peak_val),
        xytext=(0, -12),
        textcoords="offset points",
        ha="center",
        fontsize=9,
        color="red"
    )

    plt.title(f"ODMR #{index:03d}")
    plt.suptitle(f"Timestamp: {timestamp} | CPS prior: {cps:.2f}")
    plt.xlabel("Frequency (MHz)")
    plt.ylabel("Contrast (arb.)")
    plt.legend()

    fname = os.path.join(run_dir, f"odmr_{index:03d}.png")
    plt.savefig(fname, dpi=200)
    plt.close()

    return peak_freq

In [27]:
def plot_all_odmrs(run_dir, all_data):
    plt.figure(figsize=(8,6))
    
    for entry in all_data:
        freqs = entry["freqs"]
        ratio = entry["signal"] / entry["reference"]
        label = f"#{entry['index']:03d} ({entry['cps']:.1f} cps)"
        plt.plot(freqs, ratio, label=label)
    
    plt.title("All ODMRs")
    plt.xlabel("Frequency (MHz)")
    plt.ylabel("MW / Reference")
    plt.legend(fontsize=8)
    
    fname = os.path.join(run_dir, "all_odmrs.png")
    plt.savefig(fname, dpi=200)
    plt.close()

In [18]:
# Create run directory
run_dir = f"overnight_run_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
os.makedirs(run_dir, exist_ok=True)

all_odmr_data = []
num_scans = 6   # 6 hours, every 20 minutes

for idx in range(num_scans):

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # 1. Laser on
    qd.laser_on(default_config)
    time.sleep(3)

    # 2. Get counts
    counts = prog_PL.acquire() / 10000
    cps = counts / qd.max_int_time_treg / qd.min_time_tns * 1e9

    # 3. Laser off
    qd.laser_off(default_config)

    # 4. ODMR acquisition
    d = prog_ODMR.acquire()
    freqs = d.frequencies
    signal = d.signal
    reference = d.reference

    # Save CSV
    save_odmr_csv(run_dir, idx, freqs, signal, reference, cps, timestamp)

    # Save solo plot
    plot_single_odmr(run_dir, idx, freqs, signal, reference, cps, timestamp)

    # Store for combined plot
    all_odmr_data.append({
        "index": idx,
        "freqs": freqs,
        "signal": signal,
        "reference": reference,
        "cps": cps,
        "timestamp": timestamp
    })

    print(f"Completed ODMR {idx+1}/{num_scans} at {timestamp}")

    # Sleep until next scan
    time.sleep(20 * 60)

Completed ODMR 1/3 at 2026-01-25 23:01:49
Completed ODMR 2/3 at 2026-01-25 23:10:32


KeyboardInterrupt: 

In [12]:
# Save combined CSV
combined = []
for entry in all_odmr_data:
    for f, s, r in zip(entry["freqs"], entry["signal"], entry["reference"]):
        combined.append({
            "index": entry["index"],
            "timestamp": entry["timestamp"],
            "cps_prior": entry["cps"],
            "frequency_MHz": f,
            "signal": s,
            "reference": r,
            "ratio": s/r
        })

pd.DataFrame(combined).to_csv(os.path.join(run_dir, "all_odmrs.csv"), index=False)

# Plot all ODMRs
plot_all_odmrs(run_dir, all_odmr_data)